<a href="https://colab.research.google.com/github/pumazzo/ISS-Python2026/blob/main/0303_in_class_ex_pca_penguins_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐧 PCA with NumPy, Pandas & Matplotlib



## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (6, 4)


## 2. Load the dataset

We download the **Palmer Penguins** CSV directly from GitHub.
It contains morphological measurements of 344 penguins from 3 species.


In [ ]:
url = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/master/inst/extdata/penguins.csv"

df = pd.read_csv(url)

print(f"Shape: {df.shape}  →  {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


## 3. Quick exploration

Let's look at the column names, types, and basic statistics.


In [ ]:
# Column names and data types
df.dtypes


In [ ]:
# How many penguins per species?
df["species"].value_counts()


In [ ]:
# Basic statistics for numeric columns
df.describe()


## 4. Select features & remove missing values

PCA requires:
- **Numeric** features only
- **No missing (NaN) values**

We keep 4 measurements and drop any row that has a missing value in those columns.


In [ ]:
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
label_col = "species"

# Drop rows where any chosen column is NaN
clean = df[features + [label_col]].dropna().copy()

print(f"Rows before: {len(df)}  |  Rows after dropna: {len(clean)}")
clean.head()


In [ ]:
# Extract numeric matrix X and label vector y
X = clean[features].to_numpy()     # shape: (n_samples, 4)
y = clean[label_col].to_numpy()    # shape: (n_samples,)

print("X shape:", X.shape)
print("Species:", np.unique(y))


## 5. Standardize the data

PCA is **sensitive to scale**.
Bill length is in mm (small values) while body mass is in grams (large values).
Without standardization, body mass would dominate the result just because it has larger numbers.

We apply **z-score normalization**:
$$X_{\text{std}} = \frac{X - \mu}{\sigma}$$

After this, every feature has mean ≈ 0 and std ≈ 1.


In [ ]:
mu    = X.mean(axis=0)        # mean of each column
sigma = X.std(axis=0, ddof=1) # std of each column (ddof=1: sample std)

Xz = (X - mu) / sigma

print("Mean per feature (should be ~0):", Xz.mean(axis=0).round(4))
print("Std  per feature (should be ~1):", Xz.std(axis=0, ddof=1).round(4))


## 6. PCA from scratch with NumPy

### Step 1 — Covariance matrix
The covariance matrix $C$ measures how much pairs of features vary together:
$$C = \frac{1}{n-1} X_{\text{std}}^\top X_{\text{std}} \qquad \text{shape: } (4 \times 4)$$


In [ ]:
n = Xz.shape[0]
C = (Xz.T @ Xz) / (n - 1)   # equivalent to np.cov(Xz.T)

print("Covariance matrix (4x4):")
print(np.round(C, 2))


### Step 2 — Eigen-decomposition

The **eigenvectors** of $C$ are the *principal directions* (axes of maximum variance).
The **eigenvalues** tell us *how much variance* each direction captures.

We use `np.linalg.eigh` (optimized for symmetric matrices).


In [ ]:
eigvals, eigvecs = np.linalg.eigh(C)

# eigh returns values in ascending order — we want descending
idx     = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

# Explained variance ratio (what % of variance does each PC capture?)
var_ratio = eigvals / eigvals.sum()

for i, (v, r) in enumerate(zip(eigvals, var_ratio)):
    print(f"PC{i+1}:  eigenvalue = {v:.3f}   explains {r*100:.1f}% of variance")


### Step 3 — Project to 2D

We keep only the **first 2 principal components** (the two directions of highest variance)
and project all data points onto them.


In [ ]:
W2 = eigvecs[:, :2]    # principal directions, shape (4, 2)
Z2 = Xz @ W2           # projected data,       shape (n, 2)

print("Projected data shape:", Z2.shape)
print("First 3 projected points:\n", Z2[:3].round(3))


## 7. Visualize the PCA result

Each point is one penguin, projected onto the first two principal components.
Colors represent the three species.


In [ ]:
species_list = np.unique(y)
colors       = ["tab:blue", "tab:orange", "tab:green"]
markers      = ["o", "s", "^"]

fig, ax = plt.subplots()

for sp, c, m in zip(species_list, colors, markers):
    mask = (y == sp)
    ax.scatter(Z2[mask, 0], Z2[mask, 1],
               label=sp, color=c, marker=m, s=30, alpha=0.8, edgecolors="none")

pc1_pct = var_ratio[0] * 100
pc2_pct = var_ratio[1] * 100
ax.set_xlabel(f"PC1  ({pc1_pct:.1f}% variance explained)")
ax.set_ylabel(f"PC2  ({pc2_pct:.1f}% variance explained)")
ax.set_title("Palmer Penguins — PCA (NumPy)")
ax.legend()
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig("pca_penguins.png", bbox_inches="tight")
plt.show()
print("Plot saved.")


## 8. (Bonus) Interpret the principal components via loadings

A **loading** is the weight of each original feature in a principal component.
Large absolute value → that feature contributes strongly to that PC.


In [ ]:
loadings = pd.DataFrame(
    W2,
    index=features,
    columns=["PC1", "PC2"]
).round(3)

print(loadings)
